In [1]:
import pytesseract
from PIL import Image
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.probability import FreqDist
import spacy
import nltk

# Configurar o idioma para português
nltk.download('stopwords')
nltk.download('punkt')
stop_words = set(stopwords.words("portuguese"))

# Carregar modelo de linguagem do SpaCy em português
nlp = spacy.load("pt_core_news_sm")

# Função para realizar OCR em uma imagem
def perform_ocr(image_path):
    # Abrir a imagem
    image = Image.open(image_path)
    # Realizar OCR usando Tesseract
    text = pytesseract.image_to_string(image, lang='por')
    return text

# Função para realizar a sumarização de texto em português
def perform_summarization(text):
    # Tokenize o texto em frases
    sentences = sent_tokenize(text, language='portuguese')
    # Tokenize as palavras e remova as stop words
    words = [word for word in word_tokenize(text, language='portuguese') if word.lower() not in stop_words]
    # Calcule a frequência das palavras
    freq_dist = FreqDist(words)
    # Calcule a frequência máxima
    max_freq = max(freq_dist.values())
    # Normalizar as frequências
    for word in freq_dist.keys():
        freq_dist[word] = (freq_dist[word]/max_freq)
    # Calcule a pontuação das frases
    sentence_scores = {}
    for sentence in sentences:
        for word in word_tokenize(sentence.lower(), language='portuguese'):
            if word in freq_dist.keys():
                if len(sentence.split(' ')) < 30:
                    if sentence not in sentence_scores.keys():
                        sentence_scores[sentence] = freq_dist[word]
                    else:
                        sentence_scores[sentence] += freq_dist[word]
    # Obtenha as frases mais importantes
    summarized_sentences = sorted(sentence_scores, key=sentence_scores.get, reverse=True)[:3]
    summary = ' '.join(summarized_sentences)
    return summary

# Função para realizar extração de entidades
def perform_entity_extraction(text):
    # Processar o texto com SpaCy
    doc = nlp(text)
    # Extrair entidades nomeadas
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    return entities

# Caminho da imagem
image_path = "cardapio.png"

# Realizar OCR
print("Performing OCR...")
ocr_text = perform_ocr(image_path)
print("OCR Result:")
print(ocr_text)
print()

# Realizar sumarização
print("Performing Summarization...")
summary_text = perform_summarization(ocr_text)
print("Summary:")
print(summary_text)
print()

# Realizar extração de entidades
print("Performing Entity Extraction...")
entities = perform_entity_extraction(ocr_text)
print("Entities:")
for entity, label in entities:
    print(f"{entity}: {label}")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\domingos.sanches\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\domingos.sanches\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Performing OCR...
OCR Result:
CAFÉ E CIA

1035 - Café Expresso Cremoso
1096 - Café Expresso Duplo
1009 - Café com Leite (peq)

1009 - Expresso Macchiato

1036- Café Médio (leite com café)
1045 - Chá

4990 - Copo de Leite (peq)

1041 - Chocolate Quente (peq)
1003 - Chocolate Quente (grande)
1043 - Chocolate Batido (grande)
1039 - Cappuccino (pequeno)
1040 - Cappuccino (médio)

5002 - Chantily ou Mel (porção)

7,50
15,00
8,25
8,25
12,00
7,90
7,20
8,00
9,90
13,00
8,50
12,00
3,65


Performing Summarization...
Summary:


Performing Entity Extraction...
Entities:
CAFÉ: ORG
CIA: ORG
Café Expresso Cremoso: LOC
Café Expresso Duplo: PER
Café com Leite: PER
Expresso Macchiato: LOC
Café Médio: ORG
Chá

4990 - Copo de Leite: MISC
Chocolate Quente: MISC
Chocolate Quente: MISC
Chocolate Batido: LOC
Cappuccino: LOC
Cappuccino: LOC
Chantily: ORG
Mel: LOC
